In [3]:
import os
import re
from datetime import datetime
from typing import Optional, Dict, Any, List

import ollama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


# =========================================================
# PART 1: CONFIG
# =========================================================
PERSIST_DIR = "./chroma_db_promotions"
COLLECTION_NAME = "promotions"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "llama3:8b"


# =========================================================
# PART 2: SAMPLE DATA
# =========================================================
SAMPLE_PROMOS = [
    {
        "doc_id": "promo_101_p01",
        "product_id": 101,
        "product_sku": "TSR20",
        "product_name": "Rubber TSR20",
        "category": "TSR",
        "country": "Thailand",
        "promotion_id": "P01",
        "promotion_name": "Heavy Discount 3 Days",
        "promotion_detail": "Heavy discount for 3 days. Target: export customers. Condition: minimum order 10 tons.",
        "discount_type": "percentage",
        "discount_value": 15,
        "duration_days": 3,
        "start_date": "2026-01-25",
        "end_date": "2026-01-27",
        "channel": "B2B",
        "status": "active",
        "priority": 90,
    },
    {
        "doc_id": "promo_101_p02",
        "product_id": 101,
        "product_sku": "TSR20",
        "product_name": "Rubber TSR20",
        "category": "TSR",
        "country": "Thailand",
        "promotion_id": "P02",
        "promotion_name": "Heavy Discount 7 Days",
        "promotion_detail": "Heavy discount for 7 days. Condition: contract customers only. Includes free logistics within Thailand.",
        "discount_type": "percentage",
        "discount_value": 10,
        "duration_days": 7,
        "start_date": "2026-01-25",
        "end_date": "2026-01-31",
        "channel": "B2B",
        "status": "active",
        "priority": 80,
    },
    {
        "doc_id": "promo_102_p03",
        "product_id": 102,
        "product_sku": "CUPLUMP",
        "product_name": "Cup Lump",
        "category": "RAW",
        "country": "Thailand",
        "promotion_id": "P03",
        "promotion_name": "Short Flash Deal 2 Days",
        "promotion_detail": "Flash deal for 2 days. Best price for farm pickup only.",
        "discount_type": "fixed",
        "discount_value": 500,
        "duration_days": 2,
        "start_date": "2026-01-26",
        "end_date": "2026-01-27",
        "channel": "B2C",
        "status": "active",
        "priority": 70,
    },
]


# =========================================================
# PART 3: BUILD TEXT FOR EMBEDDING
# [มีแล้วใน notebook ของคุณ]
# =========================================================
def build_embedding_text(row: dict) -> str:
    return f"""
Product Name: {row.get('product_name')}
SKU: {row.get('product_sku')}
Category: {row.get('category')}
Country: {row.get('country')}

Promotion Name: {row.get('promotion_name')}
Promotion Detail: {row.get('promotion_detail')}
Discount: {row.get('discount_type')} {row.get('discount_value')}
Duration: {row.get('duration_days')} days
Channel: {row.get('channel')}
Status: {row.get('status')}
Priority: {row.get('priority')}
Start Date: {row.get('start_date')}
End Date: {row.get('end_date')}
"""


# =========================================================
# PART 4: BUILD METADATA
# [มีแล้วใน notebook ของคุณ]
# =========================================================
def build_metadata(row: dict) -> dict:
    return {
        "doc_id": row["doc_id"],
        "product_id": row["product_id"],
        "product_sku": row["product_sku"],
        "product_name": row["product_name"],
        "category": row["category"],
        "country": row["country"],
        "promotion_id": row["promotion_id"],
        "promotion_name": row["promotion_name"],
        "duration_days": row["duration_days"],
        "discount_type": row["discount_type"],
        "discount_value": row["discount_value"],
        "start_date": row["start_date"],
        "end_date": row["end_date"],
        "channel": row["channel"],
        "status": row["status"],
        "priority": row["priority"],
    }


# =========================================================
# PART 5: EMBEDDING + CHROMA INIT
# [ของเดิมคุณมีแล้ว แต่ตัด streamlit cache ออกให้ run ง่าย]
# =========================================================
def get_embeddings():
    return HuggingFaceEmbeddings(model_name=EMBED_MODEL)


def load_or_create_chroma():
    embeddings = get_embeddings()
    vectordb = Chroma(
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIR,
        embedding_function=embeddings,
    )
    return vectordb


# =========================================================
# PART 6: SEED DATA
# [ของเดิมคุณมีแล้ว]
# =========================================================
def seed_data_if_empty(vectordb: Chroma):
    try:
        got = vectordb.get()
        existing_ids = got.get("ids", [])
    except Exception:
        existing_ids = []

    if existing_ids:
        print(f"Chroma already has data: {len(existing_ids)} records")
        return

    texts = []
    metadatas = []
    ids = []

    for row in SAMPLE_PROMOS:
        texts.append(build_embedding_text(row))
        metadatas.append(build_metadata(row))
        ids.append(row["doc_id"])

    vectordb.add_texts(texts=texts, metadatas=metadatas, ids=ids)
    vectordb.persist()
    print(f"Seeded {len(ids)} promotion records into Chroma")


# =========================================================
# PART 7: QUERY PARSER
# [อันนี้ต้องเพิ่ม]
# ใช้ parse คำถาม user เป็นเงื่อนไข structured
# =========================================================
def parse_query(query: str) -> Dict[str, Any]:
    q = query.upper()

    parsed = {
        "product_sku": None,
        "duration_days": None,
        "date_start": None,
        "date_end": None,
        "raw_query": query,
    }

    # product sku
    if "TSR20" in q:
        parsed["product_sku"] = "TSR20"
    elif "CUPLUMP" in q:
        parsed["product_sku"] = "CUPLUMP"

    # duration
    m = re.search(r"(\d+)\s*day", query.lower())
    if m:
        parsed["duration_days"] = int(m.group(1))
    else:
        # เผื่อ user พิมพ์ "3 วัน"
        m_th = re.search(r"(\d+)\s*วัน", query)
        if m_th:
            parsed["duration_days"] = int(m_th.group(1))

    # date range: รองรับแบบ YYYY-MM-DD ถึง YYYY-MM-DD
    date_matches = re.findall(r"\d{4}-\d{2}-\d{2}", query)
    if len(date_matches) >= 2:
        parsed["date_start"] = date_matches[0]
        parsed["date_end"] = date_matches[1]
    elif len(date_matches) == 1:
        parsed["date_start"] = date_matches[0]
        parsed["date_end"] = date_matches[0]

    return parsed


# =========================================================
# PART 8: BUILD CHROMA FILTER
# [อันนี้ต้องเพิ่ม]
# filter เฉพาะ field ที่ Chroma filter ได้ตรงๆ
# =========================================================
def build_chroma_filter(parsed: dict):

    conditions = []

    if parsed["product_sku"]:
        conditions.append({"product_sku": parsed["product_sku"]})

    if parsed["duration_days"] is not None:
        conditions.append({"duration_days": parsed["duration_days"]})

    if not conditions:
        return None

    if len(conditions) == 1:
        return conditions[0]

    return {"$and": conditions}


# =========================================================
# PART 9: DATE OVERLAP CHECK
# [อันนี้ต้องเพิ่ม]
# เอาไว้เช็คว่าช่วงโปรโมชั่นทับกับช่วงที่ user ถามไหม
# =========================================================
def date_overlap(promo_start: str, promo_end: str, query_start: str, query_end: str) -> bool:
    ps = datetime.strptime(promo_start, "%Y-%m-%d").date()
    pe = datetime.strptime(promo_end, "%Y-%m-%d").date()
    qs = datetime.strptime(query_start, "%Y-%m-%d").date()
    qe = datetime.strptime(query_end, "%Y-%m-%d").date()

    return ps <= qe and pe >= qs


# =========================================================
# PART 10: SEARCH PROMOTIONS
# [อันนี้คือหัวใจ]
# 1) ใช้ metadata filter ก่อน
# 2) ใช้ similarity search
# 3) ค่อย date filter ซ้ำ
# =========================================================
def search_promotions(vectordb: Chroma, query: str, top_k: int = 5):
    parsed = parse_query(query)
    chroma_filter = build_chroma_filter(parsed)

    print("Parsed Query:", parsed)
    print("Chroma Filter:", chroma_filter)

    # similarity search + metadata filter
    results = vectordb.similarity_search(
        query=query,
        k=top_k,
        filter=chroma_filter
    )

    # date filter เพิ่ม
    if parsed["date_start"] and parsed["date_end"]:
        filtered_results = []
        for doc in results:
            meta = doc.metadata
            if date_overlap(
                promo_start=meta["start_date"],
                promo_end=meta["end_date"],
                query_start=parsed["date_start"],
                query_end=parsed["date_end"],
            ):
                filtered_results.append(doc)
        results = filtered_results

    # sort by priority สูงไปต่ำ
    results = sorted(results, key=lambda x: x.metadata.get("priority", 0), reverse=True)

    return results, parsed


# =========================================================
# PART 11: BUILD CONTEXT FOR LLM
# [อันนี้ต้องเพิ่ม]
# ให้ context structured ชัดๆ เพื่อกัน LLM มั่ว
# =========================================================
def build_context_from_results(results) -> str:
    if not results:
        return "No matching promotions found."

    lines = []
    for i, doc in enumerate(results, start=1):
        m = doc.metadata
        lines.append(
            f"""
Promotion #{i}
- Product SKU: {m.get('product_sku')}
- Product Name: {m.get('product_name')}
- Promotion ID: {m.get('promotion_id')}
- Promotion Name: {m.get('promotion_name')}
- Duration Days: {m.get('duration_days')}
- Discount Type: {m.get('discount_type')}
- Discount Value: {m.get('discount_value')}
- Start Date: {m.get('start_date')}
- End Date: {m.get('end_date')}
- Channel: {m.get('channel')}
- Status: {m.get('status')}
- Priority: {m.get('priority')}
- Detail: {doc.page_content}
"""
        )
    return "\n".join(lines)


# =========================================================
# PART 12: ASK LLM TO SUMMARIZE
# [แทน RetrievalQA เดิม]
# =========================================================
def ask_llm(question: str, context: str) -> str:
    prompt = f"""
You are a promotion search assistant.

Your task:
1. Answer only from the provided context.
2. Do not invent promotions.
3. If there is no matching promotion, say clearly that no matching promotion was found.
4. Summarize clearly in bullet points.
5. Mention:
   - product
   - promotion name
   - duration
   - date range
   - discount
   - important condition if available

User Question:
{question}

Context:
{context}

Answer:
"""

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]


# =========================================================
# PART 13: MAIN PIPELINE
# [อันนี้ต้องเพิ่ม]
# =========================================================
def run_pipeline(vectordb: Chroma, user_query: str, top_k: int = 5):
    results, parsed = search_promotions(vectordb, user_query, top_k=top_k)

    print(f"\nMatched documents: {len(results)}")
    for i, doc in enumerate(results, start=1):
        print(f"{i}. {doc.metadata.get('promotion_name')} | {doc.metadata.get('product_sku')}")

    context = build_context_from_results(results)
    answer = ask_llm(user_query, context)

    return {
        "parsed_query": parsed,
        "results": results,
        "context": context,
        "answer": answer,
    }


# =========================================================
# PART 14: SCRIPT ENTRY
# =========================================================
if __name__ == "__main__":
    vectordb = load_or_create_chroma()
    seed_data_if_empty(vectordb)

    test_queries = [
        "TSR20 มีโปรโมชั่น 3 วันไหม",
        "TSR20 มีโปรโมชั่น 7 วันไหม",
        "TSR20 มีโปรโมชั่น 3 วัน ช่วง 2026-01-25 ถึง 2026-01-27 ไหม",
        "CupLump มีโปรโมชั่น 2 วัน ช่วง 2026-01-26 ถึง 2026-01-27 ไหม",
        "TSR20 มีโปร flash deal ไหม",
    ]

    for q in test_queries:
        print("\n" + "=" * 80)
        print("USER QUERY:", q)

        output = run_pipeline(vectordb, q, top_k=5)

        print("\nFINAL ANSWER:")
        print(output["answer"])

Number of requested results 5 is greater than number of elements in index 3, updating n_results = 3


Chroma already has data: 3 records

USER QUERY: TSR20 มีโปรโมชั่น 3 วันไหม
Parsed Query: {'product_sku': 'TSR20', 'duration_days': 3, 'date_start': None, 'date_end': None, 'raw_query': 'TSR20 มีโปรโมชั่น 3 วันไหม'}
Chroma Filter: {'$and': [{'product_sku': 'TSR20'}, {'duration_days': 3}]}

Matched documents: 1
1. Heavy Discount 3 Days | TSR20


Number of requested results 5 is greater than number of elements in index 3, updating n_results = 3



FINAL ANSWER:
**Matching Promotion Found**

Here are the details of the promotion:

• **Product**: Rubber TSR20 (TSR20)
• **Promotion Name**: Heavy Discount 3 Days
• **Duration**: 3 days
• **Date Range**: January 25, 2026 - January 27, 2026
• **Discount**: 15% (percentage-based discount)
• **Important Condition**: Minimum order of 10 tons

Please note that this promotion is only applicable to export customers.

USER QUERY: TSR20 มีโปรโมชั่น 7 วันไหม
Parsed Query: {'product_sku': 'TSR20', 'duration_days': 7, 'date_start': None, 'date_end': None, 'raw_query': 'TSR20 มีโปรโมชั่น 7 วันไหม'}
Chroma Filter: {'$and': [{'product_sku': 'TSR20'}, {'duration_days': 7}]}

Matched documents: 1
1. Heavy Discount 7 Days | TSR20


Number of requested results 5 is greater than number of elements in index 3, updating n_results = 3



FINAL ANSWER:
Based on the provided context, I found a matching promotion for TSR20:

• **Product:** Rubber TSR20 (TSKU: TSR20)
• **Promotion Name:** Heavy Discount 7 Days
• **Duration:** 7 days
• **Date Range:** January 25-31, 2026
• **Discount:** Percentage discount of 10%
• **Important Condition:** The promotion is only available for contract customers.

No other promotions were found matching the query.

USER QUERY: TSR20 มีโปรโมชั่น 3 วัน ช่วง 2026-01-25 ถึง 2026-01-27 ไหม
Parsed Query: {'product_sku': 'TSR20', 'duration_days': 3, 'date_start': '2026-01-25', 'date_end': '2026-01-27', 'raw_query': 'TSR20 มีโปรโมชั่น 3 วัน ช่วง 2026-01-25 ถึง 2026-01-27 ไหม'}
Chroma Filter: {'$and': [{'product_sku': 'TSR20'}, {'duration_days': 3}]}

Matched documents: 1
1. Heavy Discount 3 Days | TSR20


Number of requested results 5 is greater than number of elements in index 3, updating n_results = 3



FINAL ANSWER:
Based on the provided context, here is a summary of the matching promotion:

• Product: Rubber TSR20 (TSR20)
• Promotion Name: Heavy Discount 3 Days
• Duration: 3 days
• Date Range: January 25, 2026 to January 27, 2026
• Discount: 15% (percentage discount)

Important Condition: Minimum order of 10 tons

USER QUERY: CupLump มีโปรโมชั่น 2 วัน ช่วง 2026-01-26 ถึง 2026-01-27 ไหม
Parsed Query: {'product_sku': 'CUPLUMP', 'duration_days': 2, 'date_start': '2026-01-26', 'date_end': '2026-01-27', 'raw_query': 'CupLump มีโปรโมชั่น 2 วัน ช่วง 2026-01-26 ถึง 2026-01-27 ไหม'}
Chroma Filter: {'$and': [{'product_sku': 'CUPLUMP'}, {'duration_days': 2}]}

Matched documents: 1
1. Short Flash Deal 2 Days | CUPLUMP


Number of requested results 5 is greater than number of elements in index 3, updating n_results = 3



FINAL ANSWER:
Based on the provided context, here is the summary of the matching promotion:

• **Product:** Cup Lump (CUPLUMP)
• **Promotion Name:** Short Flash Deal 2 Days
• **Duration:** 2 days
• **Date Range:** January 26-27, 2026
• **Discount:** fixed 500

No important condition is mentioned in the provided context.

USER QUERY: TSR20 มีโปร flash deal ไหม
Parsed Query: {'product_sku': 'TSR20', 'duration_days': None, 'date_start': None, 'date_end': None, 'raw_query': 'TSR20 มีโปร flash deal ไหม'}
Chroma Filter: {'product_sku': 'TSR20'}

Matched documents: 2
1. Heavy Discount 3 Days | TSR20
2. Heavy Discount 7 Days | TSR20

FINAL ANSWER:
Based on the provided context, I found a matching promotion that meets your request:

• Product: Rubber TSR20
• Promotion name: Heavy Discount 3 Days
• Duration: 3 days
• Date range: January 25, 2026 - January 27, 2026
• Discount: 15% (percentage)
• Important condition: Minimum order of 10 tons

This promotion is currently active and meets the crite